# Lecture 3: Local Explanations -- LIME & SHAP
**Question:** why did the model make *this* prediction for *this* instance?
**Data:** breast cancer (target 1 = benign) with a random forest. Cells marked `TODO` are your tasks.

In [ ]:
!pip install -q lime shap

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import itertools, math
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import shap
from lime.lime_tabular import LimeTabularExplainer
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

d = load_breast_cancer(as_frame=True); X, y = d.data, d.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
rf = RandomForestClassifier(200, random_state=0).fit(X_train, y_train)
print("test accuracy:", rf.score(X_test, y_test))
# pick the most uncertain test instance: more interesting to explain
i0 = int(np.argmin(np.abs(rf.predict_proba(X_test)[:, 1] - 0.5)))
x0 = X_test.iloc[i0]; print("P(benign) for x0:", rf.predict_proba(x0.to_frame().T)[0, 1])

## Part 1: LIME
LIME samples perturbed points around $x$, weights them by proximity, and fits a **sparse linear surrogate** to the black-box predictions.

In [ ]:
lime_exp = LimeTabularExplainer(X_train.values, feature_names=list(X.columns),
    class_names=["malignant", "benign"], mode="classification", random_state=0)
e = lime_exp.explain_instance(x0.values, rf.predict_proba, num_features=8, num_samples=5000)
print("local surrogate R^2:", e.score)
e.as_pyplot_figure(); plt.tight_layout(); plt.show()

**TODO 1 (stability).** Explain `x0` ten times with different `random_state` values for `LimeTabularExplainer`. Collect the top-5 features each time and compute the average pairwise Jaccard overlap. How stable is LIME here?

**TODO 2 (kernel width).** Repeat with `kernel_width` in `[0.5, 1, 3, 10]` (pass it to the explainer; default is $0.75\sqrt{\#features}$). How do the explanation and the surrogate $R^2$ (`e.score`) change? What does the kernel width mean?

In [ ]:
# TODO 1 / 2

## Part 2: Shapley values by hand
Features are "players", the prediction is the "payout". For a feature set $S$ we define $v(S)$ = mean prediction when features in $S$ are fixed to $x$'s values and the others are drawn from background data. We use only the 4 most important features, since exact computation needs $2^M$ evaluations.

In [ ]:
top = list(pd.Series(rf.feature_importances_, index=X.columns).nlargest(4).index); print(top)
bg = X_train.sample(100, random_state=0)

def v(S, x=x0):
    Z = bg.copy()
    for f in S: Z[f] = x[f]
    return rf.predict_proba(Z)[:, 1].mean()

def exact_shapley(feats, x=x0):
    n = len(feats); phi = {}
    for f in feats:
        others = [g for g in feats if g != f]; total = 0
        for r in range(n):
            for S in itertools.combinations(others, r):
                w = math.factorial(r) * math.factorial(n - r - 1) / math.factorial(n)
                total += w * (v(list(S) + [f], x) - v(list(S), x))
        phi[f] = total
    return phi

phi = exact_shapley(top); pd.Series(phi)

**TODO 3 (efficiency).** Check that `sum(phi.values())` equals `v(top) - v([])`. State the efficiency property in words.

**TODO 4 (sampling).** Implement the permutation-sampling estimator: for each sampled ordering of the features, add them one by one and record each feature's marginal contribution; average over orderings. Compare the error against `exact_shapley` for 10, 100 and 1000 orderings. (This is the idea behind SHAP's model-agnostic estimators.)

In [ ]:
# TODO 3 / 4

## Part 3: SHAP library
`TreeExplainer` computes Shapley values for tree ensembles exactly and fast (TreeSHAP).

In [ ]:
explainer = shap.TreeExplainer(rf)
sv = explainer(X_test)[..., 1]            # explanation for class 1 (benign)
shap.plots.waterfall(sv[i0])

In [ ]:
shap.plots.beeswarm(sv, max_display=12)
shap.plots.bar(sv, max_display=12)
shap.plots.scatter(sv[:, "worst radius"], color=sv)

**TODO 5 (additivity).** Verify for *all* test samples that `base_value + sum(shap values)` equals `rf.predict_proba(X_test)[:, 1]`.

**TODO 6 (reading plots).** From the beeswarm: which feature matters most globally? Does a high value of it push towards malignant or benign? Pick one test instance where the top feature is *not* the largest contributor and explain why.

**TODO 7 (KernelSHAP).** Use `shap.KernelExplainer(lambda z: rf.predict_proba(z)[:, 1], shap.sample(X_train, 50, random_state=0))` on 20 test samples (`nsamples=300`). Correlate its values with TreeSHAP's. How long does each take?

In [ ]:
# TODO 5 / 6 / 7

## Part 4: LIME vs SHAP
**TODO 8.** For 20 test instances, compute the top-5 features by |LIME weight| and by |SHAP value|. Report the average overlap. Examine one instance where they disagree: which explanation do you trust more, and what evidence could you use (hint: LIME's local $R^2$, SHAP's additivity)?

## Discussion
1. Why can two LIME runs on the same instance differ? Which design choices could you change to make it more stable?
2. Which Shapley axioms does LIME *not* guarantee?
3. SHAP with a background dataset: what does a SHAP value mean when features are strongly correlated?
4. Would you show a SHAP waterfall plot to a patient? What would you change?